In [0]:
Path = "/Workspace/Users/anidesifitriaaaa@gmail.com/Drafts/gold_layer_data"

df_ml = spark.read.format("delta").load(Path)

df_ml.show(10, False)

df_ml.printSchema()


In [0]:
df_ml.isEmpty()


In [0]:
from pyspark.sql.functions import col

df_ml = df_ml.drop("Patient_ID", "filename", "ingesttime", "Room_Number", "Doctor", "Date_of_Admission", "Discharge_Date", "Hospital").withColumn("Billing_Amount", col("Billing_Amount").cast("double"))
 
df_ml.show(10, False)

In [0]:
print(df_ml.dtypes)

In [0]:
train_spark, test_spark = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training Dataset Count: {train_spark.count()}")
print(f"Testing Dataset Count: {test_spark.count()}")

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor

#string indexer
categorical_columns = [field for (field, dataType) in df_ml.dtypes if dataType == "string"]

OutputIndex_cols = [x + "Index" for x in categorical_columns]

string_indexer = StringIndexer(inputCols=categorical_columns, outputCols=OutputIndex_cols, handleInvalid="skip")

numerical_columns = [field for (field, dataType) in df_ml.dtypes if dataType in ("double", "int") and field != "Billing_Amount"]

VectorAssemblerInput_cols = OutputIndex_cols + numerical_columns

vector_assembler = VectorAssembler(inputCols=VectorAssemblerInput_cols, outputCol="features")

print("Preprocessing has been completed")

In [0]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features", labelCol="Billing_Amount")

pipeline = Pipeline(stages=[string_indexer, vector_assembler, dt])

pipeline_train = pipeline.fit(train_spark)

pred_dt = pipeline_train.transform(test_spark)

pred_dt.select("features", "Billing_Amount", "prediction").show(10, False)


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="Billing_Amount", predictionCol="prediction", metricName="rmse")

rmse = evaluator.evaluate(pred_dt)

print(f"RMSE is {rmse:.2f}")

r2_dt = evaluator.setMetricName("r2").evaluate(pred_dt)
print(f"R2 is {r2_dt}")

In [0]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# 1. Define Categorical & Numerical pre-processing before linear regression (lr)
cat_cols_lr = [field for (field, dataType) in df_ml.dtypes if dataType == "string"]
num_cols_lr = [field for (field, dataType) in df_ml.dtypes if dataType in ("double", "int") and field != "Billing_Amount"]

# 2. String Indexer for Linear Regression
index_cols_lr = [x + "_IndexLR" for x in cat_cols_lr]
indexer_lr = StringIndexer(inputCols=cat_cols_lr, outputCols=index_cols_lr, handleInvalid="skip")

# 3. One Hot Encoder LR 
ohe_cols_lr = [x + "_OHE" for x in cat_cols_lr]
ohe_encoder = OneHotEncoder(inputCols=index_cols_lr, outputCols=ohe_cols_lr)

# 4. Vector Assembler LR
assembler_inputs_lr = ohe_cols_lr + num_cols_lr
# Kita beri nama kolom outputnya "features_lr" agar beda dengan Decision Tree
assembler_lr = VectorAssembler(inputCols=assembler_inputs_lr, outputCol="features_lr") 

# 5. Linear Regression
lr = LinearRegression(featuresCol="features_lr", labelCol="Billing_Amount")

# 6. Apply Pipeline
pipeline_lr = Pipeline(stages=[indexer_lr, ohe_encoder, assembler_lr, lr])

# 7. Fitting train data into lr model
PipelineModel_lr = pipeline_lr.fit(train_spark)

#8. Validation model using test data
pred_lr = pipelineModel_lr.transform(test_spark)

#9. Checking features and predictions
pred_lr.select("features_lr", "Billing_Amount", "prediction").show(10, False)




In [0]:
#Evaluating model uses RMSE, R2
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol="Billing_Amount",
    predictionCol="prediction",
    metricName="rmse")

rmse = evaluator.evaluate(pred_lr)
print(f"RMSE is {rmse:.2f}")

r2_lr = evaluator.setMetricName("r2").evaluate(pred_lr)
print(f"R2 is {r2_lr}")